# Chapter 10: Semantic Failure and Hard Negatives

Labelled failure matrix and stage classification.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


## Build a labelled query/candidate matrix

In [ ]:
cases = [
    ("battery", "no-answer overlap", "topical-but-insufficient"),
    ("R71 request", "wrong R17 instruction", "identifier confusion"),
    ("lab key", "conflicting ops note", "contradiction"),
    ("selector tail", "truncated section", "input omission"),
]
for query, candidate, label in cases:
    print(f"{query:16} -> {candidate:24} : {label}")
assert len(cases) >= 4

## Topical-but-insufficient candidate

In [ ]:
from semantic_search import DenseIndex
recs = [
    {"passage_id": "battery-note", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
idx = DenseIndex(recs, [[0.9,0.1]], "m")
q = {"values":[0.9,0.1], "model_fingerprint":"m"}
hits = idx.rank(q,1)
print(hits[0]["passage_id"])
assert hits[0]["passage_id"] == "battery-note"  # returned, yet insufficient by bundle judgement

## Wrong ID, contradiction, truncation

In [ ]:
labels = {
    "wrong_id": "identifier confusion",
    "contradiction": "conflicting evidence",
    "truncation": "input omission",
}
for k,v in labels.items():
    print(k, "->", v)
assert labels["wrong_id"] != labels["contradiction"]

## No-answer overlap

In [ ]:
# Empty bundle means no sufficient evidence, not zero recall.
from evidence_units import evidence_success
print(evidence_success({"bundles": []}, []))
assert evidence_success({"bundles": []}, []) == {"single_unit_sufficient": None, "bundle_covered": None}

## Classify by stage

In [ ]:
stage_map = {
    "vocabulary mismatch": "discovery",
    "topical neighbour": "ranking",
    "identifier confusion": "ranking",
    "contradiction": "ranking",
    "missing context": "unit boundary",
    "input omission": "encoding",
    "approximation loss": "approximate pool",
    "policy starvation": "current-view gate",
}
for failure, stage in stage_map.items():
    print(f"{failure:24} -> {stage}")
assert len(set(stage_map.values())) >= 5

## Try it yourself
Add a case or change a stage label.

In [ ]:
for name in ["topical neighbour", "no-answer confusion"]:
    print(name, "->", stage_map.get(name, "unclassified"))

## Observed versus not established
Observed: stage labels for constructed cases. Not established: general failure rates, remedies or downstream helpfulness.